In [17]:
# limpio la memoria
Sys.time()
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

[1] "2025-11-10 06:33:57 UTC"

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,783889,41.9,1454648,77.7,1454648,77.7
Vcells,1553474,11.9,1179937843,9002.3,1474910825,11252.7


In [ ]:
PARAM <- list()
PARAM$experimento <- "apo-001"
PARAM$semilla_primigenia <- 102191

In [ ]:
setwd("/content/buckets/b1/exp")
experimento_folder <- PARAM$experimento
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [19]:
Sys.time()
require( "data.table" )

# leo el dataset
dataset <- fread("~/buckets/b1/datasets/competencia_02_crudo.csv.gz" )

# calculo el periodo0 consecutivo
dsimple <- dataset[, list(
  "pos" = .I,
  numero_de_cliente,
  periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 )
]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
  shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente
]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
  ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
  clase_ternaria := "BAJA+1"
]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
  & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
  clase_ternaria := "BAJA+2"
]

# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

rm(dsimple)
gc()
Sys.time()

[1] "2025-11-10 06:36:23 UTC"

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,784833,42,1454648,77.7,1454648,77.7
Vcells,722202259,5510,1179937843,9002.3,1474910825,11252.7


[1] "2025-11-10 06:36:39 UTC"

In [3]:
#Veo si en 202006 active_quarter son todos 0
dataset[foto_mes == 202006, .N, by = active_quarter]

active_quarter,N
<int>,<int>
0,153102


In [4]:
#Veo si en 202006 internet son todos 0
dataset[foto_mes == 202006, .N, by = internet]

internet,N
<int>,<int>
0,153102


# si el campo internet invirtió sus valores en 202010

In [6]:
# 3) Matriz de transición 202009 → 202010 (más visual)

d9  <- dataset[foto_mes == 202009, .(numero_de_cliente, internet_09 = internet)]
d10 <- dataset[foto_mes == 202010, .(numero_de_cliente, internet_10 = internet)]

trans_0910 <- d9[d10, on = "numero_de_cliente", nomatch = 0L][
  internet_09 %in% 0:1 & internet_10 %in% 0:1,
  .N, by = .(internet_09, internet_10)
][order(internet_09, internet_10)]

trans_0910[, pct := N / sum(N)]
trans_0910

internet_09,internet_10,N,pct
<int>,<int>,<int>,<dbl>
0,0,8826,0.056426092
0,1,801,0.005120927
1,0,143542,0.917687975
1,1,3248,0.020765006


In [9]:
# Si flip_rate ≈ 1, se invirtió para (casi) todos.
# Si hay same > 0 o out_of_domain > 0, hay excepciones o valores raros.

# ordenar para que el lag sea del mes anterior disponible del mismo cliente

setorder(dataset, numero_de_cliente, foto_mes)

# valor previo de iternet por cliente
dataset[, internet_lag1 := shift(internet, 1L, NA, "lag"), by = numero_de_cliente]

# evaluar sólo 202010 con lag disponible y binario
eval_202010 <- dataset[
  foto_mes == 202010 & !is.na(internet) & !is.na(internet_lag1),
  .(
    N = .N,
    flips = sum(internet %in% 0:1 & internet_lag1 %in% 0:1 & internet == 1L - internet_lag1),
    same  = sum(internet %in% 0:1 & internet_lag1 %in% 0:1 & internet == internet_lag1),
    out_of_domain = sum(!(internet %in% 0:1) | !(internet_lag1 %in% 0:1))
  )
][, `:=`(flip_rate = flips/N, same_rate = same/N)]
eval_202010

In [15]:
library(data.table)

# 0) Sanidad básica
stopifnot("foto_mes" %in% names(dataset), "numero_de_cliente" %in% names(dataset))

# 1) Normalizo 'internet' a binario 0/1
#    Soporta numeric, logical, character (si/no, true/false, 1/0)
norm_internet <- function(x){
  if (is.logical(x)) return(as.integer(x))
  if (is.numeric(x)) return(as.integer(x))
  if (is.character(x)) {
    xl <- tolower(trimws(x))
    pos <- xl %in% c("1","si","sí","true","t","y","yes")
    neg <- xl %in% c("0","no","false","f","n")
    out <- rep(NA_integer_, length(x))
    out[pos] <- 1L
    out[neg] <- 0L
    return(out)
  }
  return(as.integer(x)) # fallback
}

if (!"internet" %in% names(dataset)) {
  stop("No existe la columna 'internet' (¿era 'iternet'?)")
}

dataset[, internet_bin := norm_internet(internet)]

# 2) Orden y lag por cliente
setorder(dataset, numero_de_cliente, foto_mes)
dataset[, internet_lag1 := shift(internet_bin, 1L, NA, "lag"), by = numero_de_cliente]

# 3) Diagnóstico rápido del mes
mes_obj <- 202010L

cat("Foto_mes=", mes_obj, "  filas=", dataset[foto_mes==mes_obj, .N], "\n")
cat("Tabla de valores brutos en el mes (useNA='ifany'):\n")
print(dataset[foto_mes==mes_obj, table(internet, useNA="ifany")])

cat("Tabla de valores binarizados en el mes:\n")
print(dataset[foto_mes==mes_obj, table(internet_bin, useNA="ifany")])

# 4) Métrica de flips vs lag (solo casos con 0/1 válidos y lag disponible)
eval_202010 <- dataset[
  foto_mes == mes_obj & !is.na(internet_bin) & !is.na(internet_lag1),
  .(
    N = .N,
    flips = sum(internet_bin %in% 0:1 & internet_lag1 %in% 0:1 &
                internet_bin == 1L - internet_lag1),
    same  = sum(internet_bin %in% 0:1 & internet_lag1 %in% 0:1 &
                internet_bin == internet_lag1),
    out_of_domain = sum(!(internet_bin %in% 0:1) | !(internet_lag1 %in% 0:1))
  )
]

if (nrow(eval_202010)==0 || eval_202010$N==0) {
  cat("\nNo hay pares (internet_bin, lag) válidos para", mes_obj, 
      "(¿falta mes anterior por cliente o todo es NA?).\n")
} else {
  eval_202010[, `:=`(
    flip_rate = flips / N,
    same_rate = same  / N
  )]
  cat("\nResumen flips vs lag en", mes_obj, ":\n")
  print(eval_202010)

  cat("\nMatriz de confusión (filas: lag, columnas: actual) en", mes_obj, ":\n")
  # Solo 0/1 válidos
  tab <- dataset[
    foto_mes == mes_obj & internet_bin %in% 0:1 & internet_lag1 %in% 0:1,
    table(internet_lag1, internet_bin)
  ]
  print(tab)
}


Foto_mes= 202010   filas= 159169 
Tabla de valores brutos en el mes (useNA='ifany'):
internet
     0      1      2      3 
154223   4058    802     86 
Tabla de valores binarizados en el mes:
internet_bin
     0      1      2      3 
154223   4058    802     86 

Resumen flips vs lag en 202010 :
        N  flips  same out_of_domain flip_rate  same_rate
    <int>  <int> <int>         <int>     <num>      <num>
1: 157310 144344 12078           888 0.9175768 0.07677834

Matriz de confusión (filas: lag, columnas: actual) en 202010 :
             internet_bin
internet_lag1      0      1
            0   8830    801
            1 143543   3248


# si el campo tmobile_app invirtió sus valores en 202010

In [10]:
# 3) Matriz de transición 202009 → 202010 (más visual)

d9  <- dataset[foto_mes == 202009, .(numero_de_cliente, tmobile_app_09 = tmobile_app)]
d10 <- dataset[foto_mes == 202010, .(numero_de_cliente, tmobile_app_10 = tmobile_app)]

trans_0910 <- d9[d10, on = "numero_de_cliente", nomatch = 0L][
  tmobile_app_09 %in% 0:1 & tmobile_app_10 %in% 0:1,
  .N, by = .(tmobile_app_09, tmobile_app_10)
][order(tmobile_app_09, tmobile_app_10)]

trans_0910[, pct := N / sum(N)]
trans_0910

tmobile_app_09,tmobile_app_10,N,pct
<int>,<int>,<int>,<dbl>
0,0,50164,0.31889641
0,1,2637,0.01676361
1,0,98953,0.62905184
1,1,5551,0.03528813


In [20]:
library(data.table)

# asegurate de tener dataset como data.table
setorder(dataset, numero_de_cliente, foto_mes)

# lag del mes anterior
dataset[, tmobile_app_lag1 := shift(tmobile_app, 1L, NA, "lag"), by = numero_de_cliente]

# analizamos sólo octubre 2020
res_tmobile <- dataset[
  foto_mes == 202010 & !is.na(tmobile_app) & !is.na(tmobile_app_lag1),
  .(
    N = .N,
    flips = sum(tmobile_app %in% 0:1 & tmobile_app_lag1 %in% 0:1 & tmobile_app == 1L - tmobile_app_lag1),
    same  = sum(tmobile_app %in% 0:1 & tmobile_app_lag1 %in% 0:1 & tmobile_app == tmobile_app_lag1),
    out_of_domain = sum(!(tmobile_app %in% 0:1) | !(tmobile_app_lag1 %in% 0:1))
  )
][, `:=`(
  flip_rate = flips / N,
  same_rate = same / N
)]

cat("\nResumen de cambios 0↔1 en 202010:\n")
print(res_tmobile)

cat("\nMatriz de transición (lag → actual):\n")
print(
  dataset[
    foto_mes == 202010 & tmobile_app %in% 0:1 & tmobile_app_lag1 %in% 0:1,
    table(tmobile_app_lag1, tmobile_app)
  ]
)



Resumen de cambios 0↔1 en 202010:
        N  flips  same out_of_domain flip_rate same_rate
    <int>  <int> <int>         <int>     <num>     <num>
1: 157309 101591 55718             0 0.6458054 0.3541946

Matriz de transición (lag → actual):
                tmobile_app
tmobile_app_lag1     0     1
               0 50167  2637
               1 98954  5551


In [11]:
# Si flip_rate ≈ 1, se invirtió para (casi) todos.
# Si hay same > 0 o out_of_domain > 0, hay excepciones o valores raros.

# ordenar para que el lag sea del mes anterior disponible del mismo cliente

setorder(dataset, numero_de_cliente, foto_mes)

# valor previo de tmobile_app por cliente
dataset[, tmobile_app_lag1 := shift(tmobile_app, 1L, NA, "lag"), by = numero_de_cliente]

# evaluar sólo 202010 con lag disponible y binario
eval_202010 <- dataset[
  foto_mes == 202010 & !is.na(tmobile_app) & !is.na(tmobile_app_lag1),
  .(
    N = .N,
    flips = sum(tmobile_app %in% 0:1 & tmobile_app_lag1 %in% 0:1 & tmobile_app == 1L - tmobile_app_lag1),
    same  = sum(tmobile_app %in% 0:1 & tmobile_app_lag1 %in% 0:1 & tmobile_app == tmobile_app_lag1),
    out_of_domain = sum(!(tmobile_app %in% 0:1) | !(tmobile_app_lag1 %in% 0:1))
  )
][, `:=`(flip_rate = flips/N, same_rate = same/N)]
eval_202010

# pandemia


Marcá como meses “críticos de pandemia” → 202003 a 202005

Marcá como meses “de recuperación atípica” → 202006 a 202012

Si hacés modelos o tendencia, podés:

excluirlos del entrenamiento,

o introducir una variable pandemia = 1 para esos meses,

o suavizar con medias móviles para evitar que esos saltos sesguen la tendencia.

In [13]:
#Veo si en 201905 mrentabilidad son todos 0
dataset[foto_mes == 201905, .N, by = mrentabilidad]

mrentabilidad,N
<dbl>,<int>
0,127202


In [ ]:
#Veo si en 201910 mrentabilidad son todos 0
dataset[foto_mes == 201910, .N, by = mrentabilidad]

In [ ]:
#Veo si en 202006 mrentabilidad son todos 0
dataset[foto_mes == 202006, .N, by = mrentabilidad]

In [ ]:
#Veo si en 201905 mrentabilidad_annual son todos 0
dataset[foto_mes == 201905, .N, by = mrentabilidad_annual]

In [ ]:
#Veo si en 201910 mrentabilidad_annual son todos 0
dataset[foto_mes == 201910, .N, by = mrentabilidad_annual]

In [ ]:
#Veo si en 202006 mrentabilidad_annual son todos 0
dataset[foto_mes == 202006, .N, by = mrentabilidad_annual]

In [ ]:
#Veo si en 201905 mactivos_margen son todos 0
dataset[foto_mes == 201905, .N, by = mactivos_margen]

In [ ]:
#Veo si en 201910 mactivos_margen son todos 0
dataset[foto_mes == 201910, .N, by = mactivos_margen]

In [ ]:
#Veo si en 202006 mactivos_margen son todos 0
dataset[foto_mes == 202006, .N, by = mactivos_margen]

In [ ]:
#Veo si en 201905 mpasivos_margen son todos 0
dataset[foto_mes == 201905, .N, by = mpasivos_margen]

In [ ]:
#Veo si en 201910 mpasivos_margen son todos 0
dataset[foto_mes == 201910, .N, by = mpasivos_margen]

In [ ]:
#Veo si en 202006 mpasivos_margen son todos 0
dataset[foto_mes == 202006, .N, by = mpasivos_margen]

# detectar columnas todas en 0 para un periodo

In [14]:
# Requiere data.table
library(data.table)

# dt = tu dataset (data.table) con al menos: foto_mes
# Excluí claves/no numéricas:
id_cols <- c("numero_de_cliente","foto_mes","clase_ternaria")
num_cols <- setdiff(names(dataset), id_cols)
num_cols <- num_cols[vapply(dataset[, ..num_cols], is.numeric, logical(1))]

# Conteo de no-cero por columna y mes
nz_counts <- dataset[ , lapply(.SD, function(v) sum(v != 0, na.rm = TRUE)),
                     by = .(foto_mes), .SDcols = num_cols]

# Total de filas por mes
n_by <- dataset[ , .N, by = .(foto_mes)]

# Paso a formato largo
library(data.table)
res <- melt(nz_counts, id.vars = "foto_mes",
            variable.name = "columna", value.name = "nonzero")
res <- res[n_by, on = "foto_mes"]
setnames(res, "N", "n_filas")

# Flag principal: todo cero (al menos una observación presente)
res[, all_zero := (nonzero == 0)]
res[]


foto_mes,columna,nonzero,n_filas,all_zero
<int>,<fct>,<int>,<int>,<lgl>
201901,active_quarter,122731,124273,FALSE
201901,cliente_vip,865,124273,FALSE
201901,internet,111514,124273,FALSE
201901,cliente_edad,124273,124273,FALSE
201901,cliente_antiguedad,124273,124273,FALSE
201901,mrentabilidad,123958,124273,FALSE
201901,mrentabilidad_annual,123992,124273,FALSE
201901,mcomisiones,121475,124273,FALSE
201901,mactivos_margen,119843,124273,FALSE


In [16]:
library(data.table)

# columnas numéricas (excluyendo claves y clase)
id_cols  <- c("numero_de_cliente","foto_mes","clase_ternaria")
num_cols <- setdiff(names(dataset), id_cols)
num_cols <- num_cols[vapply(dataset[, ..num_cols], is.numeric, logical(1))]

# conteo de valores no-cero por columna y mes
nz_counts <- dataset[, lapply(.SD, function(v) sum(v != 0, na.rm = TRUE)),
                     by = .(foto_mes), .SDcols = num_cols]

# total de filas por mes
n_by <- dataset[, .N, by = .(foto_mes)]

# paso a formato largo
res <- melt(nz_counts, id.vars = "foto_mes",
            variable.name = "columna", value.name = "nonzero")
res <- res[n_by, on = "foto_mes"]
setnames(res, "N", "n_filas")

# flag: todo cero (ignorando NAs)
res[, all_zero := (nonzero == 0L)]

# --- OPCIONAL: excluir casos "todo NA" (si no querés confundir)
# nn_counts <- dataset[, lapply(.SD, function(v) sum(!is.na(v))), 
#                      by = .(foto_mes), .SDcols = num_cols]
# nn_long <- melt(nn_counts, id.vars = "foto_mes",
#                 variable.name = "columna", value.name = "non_na")
# res <- res[nn_long, on = .(foto_mes, columna)]
# res[, all_zero := all_zero & (non_na > 0L)]

# SOLO imprimir los que están todo en 0
res[all_zero == TRUE, .(foto_mes, columna, n_filas)][order(foto_mes, columna)]


foto_mes,columna,n_filas
<int>,<fct>,<int>
201901,tmobile_app,124273
201901,cmobile_app_trx,124273
201901,internet_lag1,124273
201901,tmobile_app_lag1,124273
201902,tmobile_app,125401
201902,cmobile_app_trx,125401
201902,tmobile_app_lag1,125401
201903,tmobile_app,125967
201903,cmobile_app_trx,125967


# Aguinaldo

In [ ]:
library(data.table)
stopifnot(is.data.table(dataset))

# Orden por cliente y mes
setorder(dataset, numero_de_cliente, foto_mes)

# Tomo columnas monetarias / numéricas (ajustaste IPC antes, joya)
id_cols  <- c("numero_de_cliente","foto_mes","clase_ternaria")
num_cols <- setdiff(names(dataset), id_cols)
num_cols <- num_cols[vapply(dataset[, ..num_cols], is.numeric, logical(1))]

# (Opcional) concentrarnos en montos típicos: m* y tarjetas
num_cols <- intersect(num_cols, grep("^(m|Visa_m|Master_m)", num_cols, value = TRUE))

# Lag1 por cliente y Δ mensual
dataset[, paste0(num_cols, "_lag1") := shift(.SD, 1L, NA, "lag"),
        by = numero_de_cliente, .SDcols = num_cols]
dataset[, paste0(num_cols, "_delta1") := mapply(function(x, lagx) x - lagx,
        .SD, dataset[, paste0(num_cols, "_lag1"), with=FALSE]),
        .SDcols = num_cols]

delta_cols <- paste0(num_cols, "_delta1")

# Filtro filas de JUN y DIC (de todos los años)
is_jun <- dataset$foto_mes %% 100 == 6
is_dec <- dataset$foto_mes %% 100 == 12

# Mediana de Δ por variable en JUN y DIC
rank_jun <- dataset[is_jun, lapply(.SD, median, na.rm = TRUE), .SDcols = delta_cols]
rank_dec <- dataset[is_dec, lapply(.SD, median, na.rm = TRUE), .SDcols = delta_cols]

# Paso a largo y limpio nombres
melt_rank <- function(dt, etiqueta){
  out <- melt(dt, measure.vars = names(dt),
              variable.name = "var_delta", value.name = paste0("mediana_", etiqueta))
  out[, variable := sub("_delta1$", "", var_delta)]
  out[, var_delta := NULL]
  out
}

rj <- melt_rank(rank_jun, "jun")
rd <- melt_rank(rank_dec, "dic")

res <- merge(rj, rd, by = "variable", all = TRUE)

# Proporción de clientes con Δ>0 (señal de "entra plata") en JUN/DIC
p_jun <- dataset[is_jun, lapply(.SD, function(v) mean(v > 0, na.rm = TRUE)), .SDcols = delta_cols]
p_dic <- dataset[is_dec, lapply(.SD, function(v) mean(v > 0, na.rm = TRUE)), .SDcols = delta_cols]
pj <- melt_rank(p_jun, "pjun"); setnames(pj, "mediana_pjun", "prop_pos_jun")
pd <- melt_rank(p_dic, "pdic"); setnames(pd, "mediana_pdic", "prop_pos_dic")

res <- Reduce(function(a,b) merge(a,b, by="variable", all=TRUE), list(res, pj[, .(variable, prop_pos_jun)], pd[, .(variable, prop_pos_dic)]))

# Ordená por señal promedio en JUN/DIC
res[, mediana_prom := rowMeans(cbind(mediana_jun, mediana_dic), na.rm = TRUE)]
res[, prop_pos_prom := rowMeans(cbind(prop_pos_jun, prop_pos_dic), na.rm = TRUE)]
setorder(res, -mediana_prom, -prop_pos_prom)

# Mostrá el top-15 “más aguinaldo”
head(res, 15L)


In [ ]:
# Clientes que alguna vez cobraron sueldo en el banco
tienen_sueldo <- dataset[, any(mpayroll > 0, na.rm=TRUE) | any(mpayroll2 > 0, na.rm=TRUE),
                         by = numero_de_cliente][V1 == TRUE, numero_de_cliente]

dataset <- dataset[numero_de_cliente %in% tienen_sueldo]
# y volvés a correr el bloque de ranking
